# Notebook 08 — Test-Time Augmentation (TTA)

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Train a 3-fold ResNet-18 ensemble with Albumentations and evaluate **Test-Time Augmentation (TTA)** by comparing single-pass unaugmented inference against multi-view probability averaging $\hat{\mathbf{p}}_{\text{TTA}}(x) = \frac{1}{K} \sum_{k=1}^{K} \mathrm{softmax}(f_\theta(\mathcal{T}_k(x)))$.

**Learning objectives:**
- Train and save K-Fold checkpoints to portable local paths.
- Evaluate single-view clean test accuracy vs. multi-view TTA accuracy.
- Combine fold checkpoints and augmented views into a calibrated prediction.

In [ ]:
from pathlib import Path

import albumentations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision.models import resnet18

## 1  Training, Validation, and Multi-View TTA Evaluation

In [ ]:
def get_accuracy(model: nn.Module, data_loader: DataLoader, device: torch.device) -> float:
    """Compute classification accuracy (%) over an entire DataLoader."""
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0


def plot_losses(
    train_acc: list[float],
    valid_acc: list[float],
    train_loss: list[float],
    valid_loss: list[float],
) -> None:
    """Plot training and validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(train_acc, color="tab:blue", label="Train Accuracy")
    ax1.plot(valid_acc, color="tab:red", label="Validation Accuracy")
    ax1.set(title="Accuracy over Epochs", xlabel="Epoch", ylabel="Accuracy (%)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(train_loss, color="tab:blue", label="Train Loss")
    ax2.plot(valid_loss, color="tab:red", label="Validation Loss")
    ax2.set(title="Loss over Epochs", xlabel="Epoch", ylabel="Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()


def validate(
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[nn.Module, float]:
    """Run one validation epoch without gradient tracking."""
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    epoch_loss = running_loss / max(len(valid_loader), 1)
    return model, epoch_loss


def train_one_epoch(
    train_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[nn.Module, torch.optim.Optimizer, float]:
    """Run one supervised training epoch."""
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, optimizer, running_loss / max(len(train_loader), 1)


def training_loop(
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int,
    device: torch.device,
    fold_idx: int,
    checkpoint_dir: Path = Path("outputs/tta_checkpoints"),
) -> tuple[nn.Module, Path]:
    """Train one fold and return the path to the best validation checkpoint."""
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_acc = -1.0
    best_path = checkpoint_dir / f"fold_{fold_idx + 1}_best.pth"
    train_losses: list[float] = []
    valid_losses: list[float] = []
    list_train_acc: list[float] = []
    list_val_acc: list[float] = []

    for epoch in range(epochs):
        model, optimizer, train_loss = train_one_epoch(
            train_loader, model, criterion, optimizer, device
        )
        model, valid_loss = validate(valid_loader, model, criterion, device)
        train_acc = get_accuracy(model, train_loader, device=device)
        valid_acc = get_accuracy(model, valid_loader, device=device)

        print(
            f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Valid loss: {valid_loss:.4f} "
            f"| Train acc: {train_acc:.2f}% | Valid acc: {valid_acc:.2f}%"
        )
        list_train_acc.append(train_acc)
        list_val_acc.append(valid_acc)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if valid_acc >= best_acc:
            best_acc = valid_acc
            torch.save(model.state_dict(), best_path)

    plot_losses(list_train_acc, list_val_acc, train_losses, valid_losses)
    return model, best_path


def get_tta_accuracy(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> float:
    """Evaluate Test-Time Augmentation by averaging softmax probabilities across 4 views."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            views = [
                images,
                torch.flip(images, dims=[3]),
                torch.flip(images, dims=[2]),
                torch.rot90(images, k=1, dims=[2, 3]),
            ]
            avg_probs = sum(torch.softmax(model(view), dim=1) for view in views) / len(views)
            predicted = torch.argmax(avg_probs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0

## 2  Dataset, 3-Fold Training, and TTA vs. Single-Pass Comparison

In [ ]:
class DogCatDataset(Dataset):
    """Portable image classification dataset reading train/test class subfolders."""

    def __init__(self, root_dir: str | Path, transform=None) -> None:
        self.root_dir = Path(root_dir)
        self.list_images_path: list[Path] = []
        self.list_labels: list[str] = []
        self.class_to_idx = {"dogs": 0, "cats": 1}
        self.transform = transform

        for sub_dir in sorted(self.root_dir.iterdir()):
            if not sub_dir.is_dir() or sub_dir.name not in self.class_to_idx:
                continue
            for image_path in sorted(sub_dir.iterdir()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.list_images_path.append(image_path)
                    self.list_labels.append(sub_dir.name)

    def __len__(self) -> int:
        return len(self.list_images_path)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        image = cv2.imread(str(self.list_images_path[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        label = np.array(self.class_to_idx[self.list_labels[idx]], dtype=np.int64)

        if self.transform is not None:
            res = self.transform(image=image)
            image = res["image"].astype(np.float32)
        else:
            image = image.astype(np.float32) / 255.0

        image = image.transpose(2, 0, 1)
        return image, label

In [ ]:
class Net(nn.Module):
    """Lightweight ResNet-18 classifier with a 3x3 input stem for 2-class tasks."""

    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        self.model = resnet18(weights=None, num_classes=num_classes)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def get_transforms() -> tuple[albumentations.Compose, albumentations.Compose]:
    """Create augmented training pipeline and clean evaluation pipeline."""
    transforms_train = albumentations.Compose(
        [
            albumentations.HorizontalFlip(p=0.5),
            albumentations.ImageCompression(quality_range=(99, 100), p=0.5),
            albumentations.Affine(
                scale=(0.8, 1.2),
                translate_percent=(-0.2, 0.2),
                rotate=(-10, 10),
                p=0.7,
            ),
            albumentations.Normalize(),
        ]
    )
    transforms_clean = albumentations.Compose([albumentations.Normalize()])
    return transforms_train, transforms_clean


DATA_DIR = Path("../data/sample_dataset")
transforms_train, transforms_clean = get_transforms()
train_data = DogCatDataset(DATA_DIR / "train", transform=transforms_train)
test_data_clean = DogCatDataset(DATA_DIR / "test", transform=transforms_clean)
test_loader = DataLoader(test_data_clean, batch_size=8, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = KFold(n_splits=3, shuffle=True, random_state=42)
checkpoint_paths: list[Path] = []

for fold_idx, (train_idx, val_idx) in enumerate(splits.split(np.arange(len(train_data)))):
    print(f"\n=== Fold {fold_idx + 1} / 3 ===")
    train_loader = DataLoader(
        train_data, batch_size=8, sampler=SubsetRandomSampler(train_idx.tolist())
    )
    val_loader = DataLoader(train_data, batch_size=8, sampler=SubsetRandomSampler(val_idx.tolist()))
    model = Net().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    _, ckpt_path = training_loop(
        model,
        criterion,
        optimizer,
        train_loader,
        val_loader,
        epochs=3,
        device=device,
        fold_idx=fold_idx,
    )
    checkpoint_paths.append(ckpt_path)

In [ ]:
for fold_idx, ckpt_path in enumerate(checkpoint_paths, start=1):
    loaded_model = Net().to(device)
    loaded_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    single_acc = get_accuracy(loaded_model, test_loader, device)
    tta_acc = get_tta_accuracy(loaded_model, test_loader, device)
    print(f"Fold {fold_idx} | Single-view Acc: {single_acc:.2f}% | 4-View TTA Acc: {tta_acc:.2f}%")